# Day 2 | Lab 2.5: Prompt Caching Across Providers (OpenAI · Anthropic · Gemini)

**Duration:** ~1.5 hours

**Scenario.** Three production-realistic uses of caching — a banking compliance chatbot (OpenAI auto-cache), an insurance claims processor (Anthropic explicit breakpoints), and an e-commerce catalog Q&A (Gemini explicit cache). Domain mix preserved from source.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Distinguish **automatic** caching (OpenAI) from **explicit-breakpoint** caching (Anthropic, Gemini) and pick the right pattern.
2. Place **cache breakpoints** correctly in Anthropic prompts to maximize hit rate.
3. Use Gemini's **explicit cache CRUD** API (create, list, update TTL, delete) for managing large reference content.
4. Calculate the dollar savings from caching for a 50K static system + 500 user prompt.
5. Design **cache-friendly prompts** — stable prefix, dynamic suffix.
6. Recognize cache-hostile patterns (timestamp injection, randomized examples, dynamic-first system prompts) and refactor them.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies

In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q openai anthropic google-genai tiktoken


## 2. API Key Configuration

> **Setup:** Go to the 🔑 icon in Colab's left sidebar → Add your API keys → Toggle "Notebook access" ON for each key.

In [1]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv("..\\.env")
except ImportError:
    pass

for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded
ANTHROPIC_API_KEY: ✅ Loaded
GEMINI_API_KEY: ✅ Loaded


## 3. Initialize Clients & Helper Functions

We set up all three provider clients, a **token counter** (to verify we exceed minimums), and utility functions for timing and cache metric extraction.

In [2]:
import time
import json
import tiktoken
from openai import OpenAI
import anthropic
from google import genai
from google.genai import types

# --- Initialize clients ---
openai_client = OpenAI()
anthropic_client = anthropic.Anthropic()
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# --- Token counter (for pre-flight verification) ---
_enc = tiktoken.encoding_for_model("gpt-4o")  # cl100k_base — close enough for all providers

def count_tokens(text):
    """Count tokens using tiktoken (approximate, works for pre-flight checks)."""
    return len(_enc.encode(text))

print("✅ All three provider clients initialized")
print(f"✅ Token counter ready (tiktoken {tiktoken.__version__})")

✅ All three provider clients initialized
✅ Token counter ready (tiktoken 0.12.0)


In [3]:
# --- Helper: Timed API call wrapper ---
def timed_call(func, *args, **kwargs):
    """Execute a function and return (result, elapsed_seconds)."""
    start = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

# --- Helper: Extract OpenAI cache stats ---
def openai_cache_stats(response):
    """Extract caching metrics from an OpenAI chat.completions response."""
    usage = response.usage
    details = usage.prompt_tokens_details
    cached = getattr(details, 'cached_tokens', 0) if details else 0
    return {
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "cached_tokens": cached,
        "cache_hit_pct": round(cached / usage.prompt_tokens * 100, 1) if usage.prompt_tokens > 0 else 0
    }

# --- Helper: Extract Anthropic cache stats ---
def anthropic_cache_stats(response):
    """Extract caching metrics from an Anthropic messages response."""
    usage = response.usage
    return {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cache_creation_tokens": getattr(usage, 'cache_creation_input_tokens', 0) or 0,
        "cache_read_tokens": getattr(usage, 'cache_read_input_tokens', 0) or 0,
    }

# --- Helper: Display results ---
def display_result(label, stats, elapsed, response_preview=None):
    """Print formatted caching metrics."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    for k, v in stats.items():
        print(f"  {k:<30s}: {v}")
    print(f"  {'latency_seconds':<30s}: {elapsed:.2f}s")
    if response_preview:
        print(f"  {'response_preview':<30s}: {response_preview[:100]}...")
    print(f"{'='*60}")

print("✅ Helper functions defined")

✅ Helper functions defined


## 4. Build Large Reference Documents (Exceeding Provider Minimums)

Prompt caching only works when the **static prefix exceeds the provider's minimum token threshold**. This section builds realistic, production-grade reference documents and **verifies their token counts** before any API call.

| Provider | Minimum | Our Target |
|----------|:---:|:---:|
| OpenAI | 1,024 tokens | ~1,800 tokens |
| Anthropic (Haiku 4.5) | **4,096 tokens** | ~5,500 tokens |
| Gemini | 1,024 tokens | ~1,800 tokens |

In [4]:
# ============================================================
# DOCUMENT 1: Banking Compliance Manual (~1,800 tokens for OpenAI)
# Business: HDFC Bank BSA/AML compliance chatbot
# ============================================================
BANKING_COMPLIANCE_SYSTEM = """You are a senior BSA/AML compliance assistant for HDFC Bank, one of India's largest private sector banks.
Your role is to help compliance officers determine reporting requirements for financial transactions.

## Regulatory Framework
You must apply the following regulations:
- Prevention of Money Laundering Act (PMLA), 2002 — India's primary AML legislation
- RBI Master Direction on KYC — Know Your Customer norms dated May 2023
- FATF Recommendations — International AML/CFT standards (40 recommendations)
- Currency Transaction Report (CTR) thresholds: ₹10,00,000 for cash transactions
- Suspicious Transaction Report (STR) guidelines per RBI circulars
- Cross-border wire transfer rules under FEMA regulations
- Unlawful Activities Prevention Act (UAPA), 1967 — Terrorist financing provisions
- Fugitive Economic Offenders Act (FEOA), 2018 — High-value economic offences

## Transaction Monitoring Rules
1. **Cash Transactions**: Report all cash deposits/withdrawals ≥ ₹10,00,000 in a single day across all branches.
   Filing deadline: 15th of the succeeding month to FIU-IND.
2. **Wire Transfers**: Flag international transfers ≥ ₹5,00,000 for enhanced due diligence.
   SWIFT messages must include complete originator and beneficiary information per FATF Recommendation 16.
3. **Structuring Detection**: Identify patterns of transactions just below thresholds.
   Example patterns: multiple ₹9,50,000 deposits, splitting into sub-threshold amounts across branches.
   Look for deposits at different branches on the same day or across consecutive days.
4. **PEP Screening**: Apply enhanced scrutiny for Politically Exposed Persons and their family members.
   PEP database must be updated quarterly. Include domestic PEPs (senior government officials, judicial officers, senior politicians).
5. **Unusual Patterns**: Flag sudden changes in transaction volume, dormant account reactivation (no activity for 12+ months),
   rapid movement of funds (deposit followed by immediate transfer), and round-trip transactions.
6. **Third-Party Transactions**: Verify beneficial ownership for all transactions conducted on behalf of others.
   Obtain declaration of beneficial ownership for all non-individual accounts.
7. **High-Risk Geographies**: Apply additional monitoring for transactions involving FATF grey/black-listed countries.
   Current grey list: Bulgaria, Burkina Faso, Cameroon, Croatia, Congo, Haiti, Kenya, Mali, Monaco, Mozambique, Namibia,
   Nigeria, South Africa, South Sudan, Syria, Tanzania, Venezuela, Vietnam, Yemen.
8. **Trade-Based Laundering**: Watch for over/under-invoicing, multiple invoicing, misrepresentation of goods/services,
   and phantom shipments in trade finance transactions. Compare invoice values with market benchmarks.
9. **Correspondent Banking**: Enhanced due diligence for all correspondent banking relationships.
   Prohibit payable-through accounts. Annual review of all nostro/vostro relationships.
10. **Hawala/Hundi Transactions**: Watch for informal value transfer indicators — cash-intensive businesses with
    unexplained international connections, matching deposits and withdrawals in different currencies.

## Response Format
For each transaction query, provide:
- **Risk Assessment**: LOW / MEDIUM / HIGH / CRITICAL
- **Reporting Required**: Yes/No and which report (CTR, STR, or both)
- **Regulatory Basis**: Cite the specific regulation, RBI circular number, or FATF recommendation
- **Recommended Action**: What the compliance officer should do next, with timeline
- **Documentation Needed**: What records to maintain and for how long (5-year retention per PMLA)

## Escalation Matrix
- LOW risk: Standard processing, maintain records for 5 years
- MEDIUM risk: Enhanced monitoring, quarterly review by compliance team, flag in AML system
- HIGH risk: Immediate STR filing within 7 days, senior officer review within 24 hours, restrict account activity
- CRITICAL risk: Immediate STR + escalation to Principal Officer + possible account freeze + inform Enforcement Directorate

## Customer Due Diligence (CDD) Requirements
- Simplified CDD: Low-risk customers, small value accounts (balance < ₹50,000), one-time transactions < ₹50,000
- Standard CDD: Regular customers, PAN/Aadhaar verification, proof of address, recent photograph
- Enhanced CDD: High-risk customers, PEPs, high-value transactions (> ₹50 lakh), non-face-to-face customers,
  NRI/foreign national accounts, trust/charity/NPO accounts, clients of special categories (DNFBP)
- Ongoing CDD: Continuous monitoring of customer activity, periodic profile updates (2 years for high-risk, 5 years for low-risk)

## Red Flags for Suspicious Activity
- Multiple cash deposits across different branches on the same day totalling near or above ₹10 lakh
- Frequent round-amount transactions (₹1,00,000, ₹2,00,000, ₹5,00,000) with no business rationale
- Transactions inconsistent with customer's declared income, business profile, or KYC documentation
- Use of shell companies, layered corporate structures, or nominee shareholders to obscure beneficial ownership
- Immediate transfer of funds after deposit (pass-through activity) especially to foreign jurisdictions
- Reluctance to provide KYC documentation, beneficial ownership information, or source of funds declaration
- Transactions involving cryptocurrency exchanges without clear business rationale or regulatory approval
- Rapid succession of loan applications, credit card applications, or account openings across group entities
- Cash-intensive business accounts with unusually high volumes compared to industry norms (e.g., jewellers, real estate)
- Frequent changes to account signatories, addresses, or contact details without apparent business reason
- Use of walk-in customers or agents to conduct transactions to avoid identification requirements
- Accounts that receive many small credits and immediately wire funds to a foreign country

## Penalty Framework (for reference only)
- Non-compliance with PMLA: Up to 3 years imprisonment, fine up to ₹5 lakh per violation
- RBI penalties: Up to ₹1 crore for KYC non-compliance, public disclosure of penalty
- FATF sanctions: Potential grey-listing impacts on correspondent banking relationships
"""

token_count = count_tokens(BANKING_COMPLIANCE_SYSTEM)
print(f"📏 Banking Compliance Document: {token_count} tokens")
print(f"   OpenAI minimum: 1,024 → {'✅ EXCEEDS' if token_count >= 1024 else '❌ TOO SHORT'}")

📏 Banking Compliance Document: 1286 tokens
   OpenAI minimum: 1,024 → ✅ EXCEEDS


In [5]:
# ============================================================
# DOCUMENT 2: Insurance Claims + Healthcare Protocol (~5,500 tokens for Anthropic)
# Business: Star Health Insurance claims triage with clinical guidelines
# Anthropic claude-haiku-4-5 requires >= 4,096 tokens minimum!
# ============================================================
INSURANCE_HEALTHCARE_SYSTEM = """You are a senior claims adjudicator and clinical reviewer for Star Health Insurance,
India's largest standalone health insurance company. You evaluate insurance claims by cross-referencing
policy terms with clinical medical guidelines. Your role requires detailed knowledge of both insurance
policy mechanics and clinical medicine to ensure accurate, fair, and timely claims processing.

## PART A: INSURANCE POLICY TERMS AND CONDITIONS

### 1. Coverage Plans — Detailed Benefits Matrix
- **Gold Plan**: ₹10,00,000 sum insured per annum per individual
  Network: 14,000+ cashless hospitals across India including all major chains (Apollo, Fortis, Max, Medanta, Narayana Health)
  Room category: Single private AC room (up to ₹8,000/day in metros, ₹5,000/day in tier-2)
  Co-pay: None for network hospitals; 20% co-pay for non-network hospitals
  Ambulance: ₹3,000 per hospitalization for road ambulance; ₹50,000 for air ambulance (emergency only)
  Annual health check: Complimentary for policyholder and spouse after 2 years of continuous renewal
  Restore benefit: 100% sum insured restored once per policy year if fully exhausted
  No-claim bonus: 10% cumulative increase per claim-free year (maximum 50% of base sum insured)
  Pre-hospitalization: 60 days | Post-hospitalization: 90 days | Day-care procedures: All 500+ listed

- **Silver Plan**: ₹5,00,000 sum insured per annum per individual
  Network: 10,000+ cashless hospitals
  Room category: Twin-sharing room (up to ₹4,000/day in metros, ₹2,500/day in tier-2)
  Co-pay: 10% on all claims regardless of hospital network status
  Ambulance: ₹2,000 per hospitalization for road ambulance only
  Annual health check: Not included
  Restore benefit: 50% of sum insured restored once per policy year
  No-claim bonus: 5% cumulative per claim-free year (maximum 25%)
  Pre-hospitalization: 30 days | Post-hospitalization: 60 days | Day-care: Listed 350+ procedures

- **Bronze Plan**: ₹3,00,000 sum insured per annum per individual
  Network: Reimbursement only (no cashless facility)
  Room category: General ward (up to ₹2,000/day)
  Co-pay: 20% on all claims plus room rent capping deductions
  Ambulance: ₹1,500 per hospitalization
  Restore benefit: Not available
  No-claim bonus: Not available
  Pre-hospitalization: 30 days | Post-hospitalization: 60 days | Day-care: Listed 200+ procedures

- **Platinum Plan**: ₹25,00,000 sum insured per annum per individual
  Network: 14,000+ cashless + international emergency cover (up to $50,000 per incident)
  Room category: Single deluxe room (no daily sub-limit)
  Co-pay: None | Ambulance: ₹5,000 road + ₹1,00,000 air ambulance
  Annual health check: Comprehensive executive health package for family (after 1 year)
  Restore benefit: 200% of sum insured (full sum restored twice per year)
  No-claim bonus: 15% cumulative per year (maximum 75%)
  Additional: Organ donor expenses up to ₹5,00,000; second opinion from empanelled specialists
  Domiciliary hospitalization: Covered if hospitalization not possible due to bed unavailability
  Pre-hospitalization: 90 days | Post-hospitalization: 180 days | Day-care: All listed + unlisted

- **Family Floater**: Available across all plan tiers
  Maximum 6 members (proposer + spouse + 4 dependent children up to age 25)
  Individual claim limit: 50% of total sum insured per member per policy year
  Additional premium: 10% per additional child beyond 2; parents/in-laws require separate policy
  Maternity sub-limit: ₹50,000 normal delivery / ₹75,000 C-section (Gold/Platinum only, 3-year wait)
  Newborn cover: From day 1 for 90 days post-delivery (must add to policy within 90 days)

### 2. Pre-Authorization Rules (Detailed)
- **Planned/Elective hospitalization**: 72-hour advance pre-authorization mandatory via Star Health portal
  Required documents: Doctor's referral letter, estimated cost breakdown, treatment plan
  Pre-auth response SLA: 4 hours during business hours (9 AM - 9 PM), 6 hours outside
  Pre-auth validity: 30 days from approval date; must notify if surgery date changes
- **Emergency admissions**: Pre-auth within 24 hours of admission; retroactive approval considered
  Emergency defined as: Condition requiring immediate hospitalization to prevent death, serious impairment,
  or serious dysfunction of any body organ or part (per IRDAI guidelines)
  If pre-auth delayed beyond 24h: Claim converted to reimbursement (not rejected)
- **Day-care procedures**: Pre-auth required for procedures estimated > ₹25,000
  Day-care defined as: Procedures requiring less than 24 hours of hospitalization due to advancement in
  medical technology (e.g., dialysis, chemotherapy, radiotherapy, lithotripsy, cataract, angiography)
- **Maternity claims**: Pre-auth required at 30 weeks; delivery hospital must be network
- **Organ transplant**: Mandatory pre-auth from Star Health chief medical officer
  Required: Organ allocation letter from NOTTO, estimated cost, donor fitness certificate
  Donor expenses covered: Harvesting, transportation, post-operative care up to 30 days
- **Cancer treatment**: Pre-auth for each chemo/radiation cycle
  Must include: Oncologist treatment protocol, TNM staging, NCCN/ESMO guideline reference
  Targeted therapy/immunotherapy: Pre-auth with molecular profiling report mandatory

### 3. Exclusion Criteria (Comprehensive List)
- Pre-existing conditions (PED): 4-year waiting period (2 years for declared PEDs on Gold/Platinum)
  Definition: Any condition diagnosed, treated, or for which medical advice/medication was received
  within 48 months before policy inception date
  Proof of PED: Medical records, pharmacy records, diagnostic reports from any source
  Undeclared PED: Claim rejection + possible policy cancellation + premium forfeiture
- Cosmetic/aesthetic procedures: Not covered unless post-accident reconstruction certified by plastic surgeon
- Self-inflicted injuries: Excluded from all plans; includes substance abuse, alcohol intoxication injuries
- Adventure sports: Covered only with rider (bungee, paragliding, scuba diving, mountaineering above 3000m)
- Dental: Excluded unless hospitalization > 24 hours required (e.g., jaw fracture, oral cancer surgery)
- Congenital conditions: Covered after 4-year continuous renewal, ₹5 lakh sub-limit per policy year
- Infertility/IVF: Not covered; optional rider ₹3 lakh limit (available on Gold/Platinum only)
- War/terrorism: Covered only under Platinum with terrorism rider
- Experimental/unproven: Not covered unless ICMR-approved clinical trial with ethics committee clearance
- AYUSH (Ayurveda, Yoga, Unani, Siddha, Homeopathy): Up to ₹50,000/year on Gold/Platinum only
  Must be at government-recognized AYUSH hospital; treatment must be by qualified BAMS/BHMS practitioners
- Hearing aids, spectacles, contact lenses: Not covered unless accidental injury necessitates
- Bariatric surgery: Only if BMI > 40 with 2+ documented comorbidities (DM, HTN, OSA, OA)
  Pre-auth mandatory with 6 months of documented failed conservative management
- Vaccination: Not covered except post-exposure prophylaxis (rabies, tetanus, hepatitis B post-needlestick)
- Stem cell therapy: Not covered unless for approved indications (certain blood cancers, aplastic anemia)
- Robotic surgery premium: Additional cost over standard procedure is patient's responsibility
  Exception: Platinum plan covers robotic surgery at par with standard rates at approved centers

### 4. Waiting Periods
- Initial: 30 days from policy start (no claims except accidental injury hospitalization)
- Specific diseases (2-year wait): Hernia, hydrocele, cataract, benign prostatic hypertrophy,
  joint replacement (hip/knee), kidney stones, gallstones, fistula, piles/hemorrhoids,
  sinusitis, tonsillectomy, adenoidectomy, pilonidal sinus, varicose veins, deviated nasal septum,
  uterine fibroids, dysfunctional uterine bleeding, endometriosis, ovarian cysts
- Maternity: 3 years continuous coverage for normal delivery and C-section
- Pre-existing conditions: 4 years (2 years for declared PEDs on Gold/Platinum)
- Mental illness: 2-year waiting period, sub-limit ₹2 lakh/year, inpatient > 24 hours only

### 5. Fraud Detection Indicators (Red Flag System)
Score each indicator 1-3 points. If total score ≥ 5, trigger mandatory investigation:
- Claims within 90 days of inception for conditions with waiting periods (3 points — HIGH)
- Multiple claims for same diagnosis within 6 months at different hospitals (3 points — HIGH)
- Non-network hospital for clearly planned/elective procedures (2 points — MEDIUM)
- Billing > 2× CGHS/AIIMS benchmark rates for the city and procedure (2 points — MEDIUM)
- Diagnosis mismatch: Admission diagnosis significantly different from discharge summary (2 points)
- Patient discharged AMA (Against Medical Advice) with full claim submitted (2 points)
- Altered/photocopied medical reports — verify originals with hospital medical records (3 points — HIGH)
- Same patient ID used at multiple hospitals concurrently (3 points — HIGH)
- Referral pattern: Single doctor referring > 20 claims/quarter to specific hospital (2 points)
- Phantom procedures: Billed procedures not matching nursing notes, OT register, or anesthesia records (3 points)
- Room upgrade charges without medical justification (ICU when ward appropriate) (1 point)
- Investigation fee: Field investigation by TPA costs ₹5,000-₹15,000, deducted from settlement if fraud confirmed

### 6. Settlement Workflow (Detailed SLAs)
1. Claim received → Initial document completeness check (2-hour SLA from receipt)
   Documents needed: Claim form, policy copy, ID proof, hospital discharge summary,
   itemized bill with procedure codes, diagnostic reports, prescription copies, pre-auth letter
2. Policy validation (4 hours): Active status, premium paid, waiting periods, sum insured availability,
   room category eligibility, sub-limits, co-pay calculation
3. Medical review by empanelled doctor (24 hours): Clinical necessity assessment,
   procedure appropriateness per clinical guidelines, length of stay justification,
   drug prescription review, implant/device necessity review
4. Cost benchmarking (8 hours): Compare billed amounts against CGHS rates, AIIMS rates,
   and Star Health's internal city-wise benchmark database
   Proportionate deduction formula: (Actual room rent / Eligible room rent) × Total bill
5. Investigation trigger: If fraud score ≥ 5 → field investigation (72 hours additional SLA)
   Investigation report must include: Hospital visit, patient interview, doctor verification, records audit
6. Settlement decision: Approve Full / Approve Partial (with itemized deductions) / Reject (with reasons)
   Deduction categories: Room rent capping, non-payable items (IRDAI list), co-pay, sub-limit excess
7. Communication to policyholder: Within 4 hours of settlement decision via SMS + email
   Rejection letter must cite: Specific policy clause, medical rationale, relevant IRDAI regulation
8. Payment: 7 working days for cashless (direct to hospital), 14 working days for reimbursement (to policyholder)
9. Grievance/appeal: 30 days from rejection; policyholder may submit additional documents
   Escalation: Star Health internal grievance → IRDAI Bima Bharosa portal → Insurance Ombudsman → Consumer Court

## PART B: CLINICAL MEDICAL GUIDELINES FOR CLAIMS REVIEW

### 1. Cardiac Emergency Protocol
- **STEMI (ST-Elevation Myocardial Infarction)**:
  Door-to-balloon time: < 90 minutes (< 60 minutes if transfer from non-PCI center)
  Activate cath lab immediately upon ECG confirmation of ST elevation in 2+ contiguous leads
  Treatment protocol: Dual antiplatelet therapy — Aspirin 325mg chewed + Ticagrelor 180mg loading
  (Alternative: Clopidogrel 600mg loading if Ticagrelor contraindicated or unavailable)
  Anticoagulation: Unfractionated heparin 60 U/kg bolus (max 4000U) + 12 U/kg/hr infusion
  Post-PCI: Aspirin 75mg daily indefinitely + Ticagrelor 90mg BID for 12 months (minimum)
  Expected hospitalization: 3-5 days uncomplicated PCI, 7-10 days if CABG required
  ICU requirement: Minimum 24-48 hours post-PCI in CCU with continuous telemetry monitoring
  Standard cost benchmarks (metro India):
    Single stent PCI: ₹2,50,000 - ₹4,50,000 (drug-eluting stent ₹30,000-₹1,20,000 each)
    Multi-vessel PCI (2-3 stents): ₹4,00,000 - ₹7,00,000
    STENT COST CAP: NPPA ceiling prices apply — verify stent pricing against NPPA list
  Red flags: Stent cost > NPPA cap, >3 stents without documented rationale, no pre-PCI angiography images

- **NSTEMI (Non-ST-Elevation MI)**:
  Risk stratification: GRACE score mandatory for treatment pathway decision
  Low risk (GRACE < 109): Conservative management, serial troponins at 0, 3, 6 hours
    Stress test before discharge, early outpatient follow-up with cardiology
  Intermediate risk (GRACE 109-140): Early invasive strategy within 72 hours preferred
  High risk (GRACE > 140): Urgent invasive strategy within 24 hours mandatory
  Expected hospitalization: 4-7 days | Standard cost (metro): ₹1,50,000-₹3,00,000

- **Heart Failure Exacerbation (ADHF)**:
  Initial management: IV Furosemide 40-80mg bolus; BiPAP if SpO2 < 92% on O2
  Workup: BNP/NT-proBNP, echocardiography, chest X-ray, renal function, electrolytes
  Classify: HFrEF (EF < 40%) vs HFpEF (EF ≥ 50%) vs HFmrEF (EF 40-49%)
  Expected hospitalization: 5-7 days, ICU 2-3 days if requiring inotropic support
  Standard cost: ₹1,00,000-₹2,50,000 | ICD/CRT implantation: ₹8,00,000-₹15,00,000 additional

### 2. Stroke Protocol (Code Stroke)
- **Ischemic Stroke**: CT head within 25 minutes of arrival (door-to-CT target)
  NIHSS (National Institutes of Health Stroke Scale) scoring mandatory for all stroke patients
  **tPA (Alteplase) eligibility**: 0-4.5 hour window from last-known-well time
    Dose: 0.9 mg/kg (max 90mg), 10% bolus IV over 1 minute, 90% infusion over 60 minutes
    Contraindications: Active internal bleeding, recent surgery within 14 days, INR > 1.7,
    platelet count < 100,000/μL, BP > 185/110 despite treatment, blood glucose < 50 mg/dL,
    prior stroke within 3 months, head trauma within 3 months
  **Mechanical thrombectomy**: For large vessel occlusion (LVO) detected on CTA/MRA
    Extended window: Up to 24 hours with favorable perfusion imaging (DAWN/DEFUSE-3 criteria)
    Standard cost: ₹5,00,000-₹10,00,000 (includes thrombectomy device + ICU + imaging)
  Expected hospitalization: 7-14 days including initial rehabilitation
  Rehabilitation: Should start within 48 hours of stabilization (physiotherapy, speech therapy, OT)

- **Hemorrhagic Stroke**: Reverse anticoagulation immediately, BP target SBP < 140 mmHg
  Neurosurgery consult within 1 hour for all cases
  EVD (External Ventricular Drain) if hydrocephalus develops; decompressive craniectomy if herniation
  Expected hospitalization: 14-21 days, ICU minimum 5-7 days
  Standard cost: ₹5,00,000-₹15,00,000 (depends on surgical intervention needed)

### 3. Surgical Procedures — Standard Duration, Stay, and Cost Benchmarks (Metro India)
| Procedure | Typical Stay | Cost Range | Notes |
|-----------|-------------|------------|-------|
| Lap Cholecystectomy | 1-2 days | ₹80,000-₹1,50,000 | Day-care in some centers |
| Appendectomy (emergency) | 2-3 days | ₹70,000-₹1,20,000 | Open vs lap cost difference ₹20,000 |
| Knee Replacement (unilateral TKR) | 5-7 days | ₹2,50,000-₹4,00,000 | Implant ₹80,000-₹2,00,000 |
| Knee Replacement (bilateral) | 7-10 days | ₹4,50,000-₹7,00,000 | Staged if comorbidities present |
| Hip Replacement (THR) | 7-10 days | ₹3,00,000-₹5,00,000 | Cemented vs uncemented implant |
| Cataract (Phaco + IOL) | Day-care | ₹25,000-₹60,000/eye | Premium IOL (toric/multifocal) costs extra |
| Hernia Repair (lap) | 1-2 days | ₹60,000-₹1,00,000 | Mesh cost ₹8,000-₹30,000 |
| CABG (coronary bypass) | 10-14 days | ₹3,50,000-₹6,00,000 | On-pump vs off-pump |
| Spinal Fusion | 5-10 days | ₹3,00,000-₹7,00,000 | Per level; implant cost major variable |
| Lap Hysterectomy | 3-5 days | ₹1,00,000-₹2,00,000 | Robotic adds ₹50,000-₹1,00,000 |
| Dialysis (per session) | Day-care | ₹2,500-₹5,000 | HD; CAPD/PD has different costing |
| Chemotherapy (per cycle) | Day-care/1 day | ₹10,000-₹2,00,000 | Varies widely by regimen |
| Liver Transplant | 21-30 days | ₹25,00,000-₹35,00,000 | Living donor; cadaveric may differ |
| Kidney Transplant | 14-21 days | ₹8,00,000-₹12,00,000 | Excludes immunosuppression cost |
| Cochlear Implant | 3-5 days | ₹6,00,000-₹10,00,000 | Device cost ₹4,00,000-₹8,00,000 |

### 4. Common Drug Interactions (Flag in Claims Review)
When reviewing prescription data in claims, flag these critical interactions immediately:
- **Warfarin + NSAIDs (Ibuprofen, Diclofenac, Aspirin)**: Synergistic bleeding risk (GI bleed, intracranial)
  Action: Flag for medical review; verify INR monitoring orders and GI prophylaxis prescribed
- **ACE Inhibitors/ARBs + Potassium supplements/Spironolactone**: Hyperkalemia risk (potentially fatal)
  Action: Verify serum K+ monitoring; check renal function before co-prescribing
- **Metformin + IV Contrast**: Lactic acidosis risk; hold metformin 48h post-contrast
  Action: Verify hold order in medication administration records; check renal function post-contrast
- **SSRIs (Fluoxetine/Sertraline) + Tramadol**: Serotonin syndrome risk (life-threatening)
  Action: Should NOT be co-prescribed; if found, flag as clinical inappropriateness
- **Clopidogrel + Omeprazole**: Omeprazole reduces Clopidogrel efficacy via CYP2C19 inhibition
  Action: Should use Pantoprazole or Rabeprazole instead; flag if Omeprazole prescribed with Clopidogrel
- **Fluoroquinolones + Corticosteroids**: Tendon rupture risk, especially elderly > 60 years
  Action: Flag in geriatric claims; verify informed consent and monitoring documentation
- **Digoxin + Amiodarone**: Amiodarone increases Digoxin levels 70-100%; toxicity risk
  Action: Digoxin dose should be reduced by 50% when Amiodarone added; verify dose adjustment
- **Statins (Atorvastatin/Rosuvastatin) + Macrolide antibiotics (Azithromycin/Clarithromycin)**: Rhabdomyolysis risk
  Action: Verify CK levels monitored; temporary statin hold during antibiotic course preferred
- **Lithium + NSAIDs/ACE Inhibitors/Diuretics**: Lithium toxicity risk
  Action: Verify serum lithium levels monitored; dose adjustment may be required
- **Insulin + Beta-Blockers**: Masked hypoglycemia symptoms; prolonged hypoglycemia episodes
  Action: Verify glucose monitoring frequency adequate; cardioselective beta-blockers preferred

### 5. Pediatric Considerations (Age-Adjusted Review)
All pediatric claims MUST include child's weight in admission notes for dosing verification:
- **Paracetamol**: 15 mg/kg q4-6h (max 60 mg/kg/day or 4g/day whichever is lower)
  Flag if: Total daily dose exceeds weight-appropriate maximum, or if given to neonates < 28 days
- **Ibuprofen**: 10 mg/kg q6-8h (only if age > 6 months; contraindicated if dehydrated/renal impairment)
- **IV fluid bolus (shock)**: 20 mL/kg isotonic crystalloid (NS or RL), may repeat up to 60 mL/kg in first hour
  Reassess after each bolus: heart rate, capillary refill, BP, urine output
- **Febrile seizure**: Generally benign if: Age 6 months - 5 years, generalized tonic-clonic, < 15 minutes
  duration, no focal features, single episode in 24 hours. Investigate if outside these parameters.
- **Neonatal jaundice**: Phototherapy thresholds per Bhutani nomogram based on age in hours and risk factors
  Exchange transfusion: If bilirubin rises despite phototherapy or exceeds exchange threshold
- **Pediatric sepsis**: Weight-based antibiotic dosing; Ceftriaxone 50-100 mg/kg/day (max 4g)
  Fluid bolus: 20 mL/kg in first 15-60 minutes; vasopressors (Dopamine 5-20 mcg/kg/min) if fluid-refractory
- **Common pediatric claim red flags**: Adult doses in pediatric prescriptions, procedures rarely done in children
  at the claimed age, hospitalization > 5 days for conditions typically managed as outpatient

### 6. Oncology Claims Review Guidelines
- Staging documentation: TNM staging report mandatory for all cancer claims
- Treatment protocol: Must reference NCCN, ESMO, or ICMR guidelines; non-standard protocols require
  tumor board recommendation letter with at least 3 oncologist signatures
- Chemotherapy: Verify regimen, number of planned cycles, body surface area (BSA) for dose calculation
  Expensive biologics (Trastuzumab, Pembrolizumab, Nivolumab): Pre-auth per cycle with response assessment
- Radiation therapy: Verify fractionation schedule (e.g., 25 fractions for breast, 33 for prostate)
  IMRT/IGRT/SBRT: Cost 2-3× conventional radiation; pre-auth with radiation oncologist justification
- Surgical oncology: Verify margins, lymph node dissection, frozen section reports
  Reconstruction after mastectomy: Covered under Gold/Platinum as medically necessary
- Palliative care: Covered for terminal illness with < 6 months prognosis (hospice care sub-limit ₹2 lakh)

### 7. Mental Health Coverage Guidelines (Mental Healthcare Act, 2017 Compliance)
- Inpatient psychiatric admission: Covered after 2-year waiting period, sub-limit ₹2 lakh/policy year
- Covered conditions: Major depressive disorder, bipolar I and II, schizophrenia spectrum disorders,
  obsessive-compulsive disorder, post-traumatic stress disorder, generalized anxiety disorder,
  panic disorder, eating disorders (anorexia nervosa, bulimia) requiring hospitalization
- Day-care psychiatric: ECT covered at all plan levels; rTMS covered under Gold/Platinum only
- Substance abuse rehabilitation: NOT covered under standard plans (per IRDAI guidelines)
- Minimum hospitalization requirement: Inpatient admission > 24 hours for claim eligibility
- Required documentation: Psychiatrist's diagnosis per ICD-11, detailed treatment plan, nursing notes,
  mental status examination (MSE) at admission and discharge, medication chart with response assessment
- Outpatient psychiatric follow-up: NOT covered (only inpatient claims eligible)
- Involuntary admission claims: Must include Form III (independent assessment) per MHA 2017

Always cite the specific policy section number, clinical guideline, IRDAI regulation,
or cost benchmark in your response. Flag any drug interactions, billing anomalies,
or fraud indicators immediately with the appropriate fraud score from the red flag system above.
"""

token_count = count_tokens(INSURANCE_HEALTHCARE_SYSTEM)
print(f"📏 Insurance + Healthcare Document: {token_count} tokens")
print(f"   Anthropic Haiku 4.5 minimum: 4,096 → {'✅ EXCEEDS' if token_count >= 4096 else '❌ TOO SHORT'}")
print(f"   OpenAI minimum: 1,024 → {'✅ EXCEEDS' if token_count >= 1024 else '❌ TOO SHORT'}")

📏 Insurance + Healthcare Document: 5606 tokens
   Anthropic Haiku 4.5 minimum: 4,096 → ✅ EXCEEDS
   OpenAI minimum: 1,024 → ✅ EXCEEDS


In [6]:
# ============================================================
# DOCUMENT 3: E-Commerce Catalog (~1,800 tokens for Gemini)
# Business: Flipkart customer support with product catalog
# ============================================================
ECOMMERCE_CATALOG = """You are a customer support AI for Flipkart, India's leading e-commerce marketplace.
Use the following product catalog and policy information to answer customer queries accurately.

## Product Categories & Return Policies
### Electronics
- Smartphones: 7-day replacement guarantee, 1-year manufacturer warranty, screen damage excluded
- Laptops: 10-day replacement, 1-year warranty + optional Flipkart Complete Protection (₹1,499-₹4,999)
- Headphones/Earbuds: 7-day replacement, 6-month warranty, hygiene-related returns not accepted
- Smart TVs: 10-day replacement, 2-year panel warranty, installation by certified technicians included
- Cameras & Drones: 7-day replacement, 1-year warranty, registration required for drones > 250g (DGCA rules)
- Gaming Consoles: 10-day replacement, 1-year warranty, digital content purchases non-refundable
- Return condition: Original packaging required, no physical damage, all accessories/manuals included

### Fashion & Apparel
- Clothing: 30-day easy return, no questions asked for size/fit issues, free return pickup
- Footwear: 30-day return, must be unworn with original tags, sole must show no wear marks
- Watches: 10-day return for non-premium, seal must be intact for premium brands (Titan, Fossil, Casio)
- Jewellery: 7-day return for gold/silver (BIS hallmark verification), 30 days for fashion jewellery
- Ethnic Wear (Saree, Lehenga): 30-day return, unstitched fabric cannot be returned once cut
- Return condition: All tags attached, unwashed, unworn, original packaging preferred

### Home & Furniture
- Large Appliances (AC, Fridge, Washing Machine): Installation by Flipkart-certified technicians within 48 hours
  Demo provided at installation, warranty card activated upon installation, return only for DOA (dead on arrival)
- Furniture: 7-day return for manufacturing defects only, assembly service available (₹499-₹1,499)
  Customer must provide lift/staircase access, extra charges for floor > 3 without lift
- Kitchen Appliances: 7-day replacement, 1-year warranty, defective units collected at doorstep
- Home Decor: 7-day return if damaged during delivery, customized/personalized items non-returnable
- Mattresses: 100-night trial for select brands (Wakefit, SleepyCat), compressed mattresses need 48h to expand

### Grocery & Fresh (Flipkart Minutes / Quick Commerce)
- Delivery: 10-30 minutes in select pin codes (metro cities, 6 AM to midnight)
- Returns: Only for damaged/expired items, report within 24 hours with photo proof
- No return on: Perishables consumed, opened sealed packs (milk, juice), frozen items once thawed
- COD not available for grocery orders < ₹200

## Delivery Partners & SLAs
- Ekart (in-house): Metro cities 1-2 days, Tier 2 cities 2-4 days, Rural areas 5-7 days
- Third-party sellers: SLA as per seller profile (Standard = 5-7 days, Premium = 2-3 days)
- Flipkart Plus: Free next-day delivery for Plus members on eligible items (metro only)
- Same-day delivery: Available in 8 cities for orders placed before 12 PM (₹99 fee, free for Plus)
- Open-box delivery: Available for electronics > ₹5,000, customer inspects before accepting
- Installation delivery: Large appliances include installation, scheduled within 48 hours of delivery

## Payment Methods
- UPI: PhonePe (Flipkart group), Google Pay, Paytm, BHIM — instant confirmation, ₹1 lakh limit per txn
- Credit/Debit Cards: Visa, Mastercard, RuPay — EMI available on orders > ₹3,000
  No-cost EMI on select products with partner bank cards (HDFC, ICICI, SBI, Kotak, Axis)
- Flipkart Pay Later: Credit line up to ₹1,00,000 for eligible users, bill due on 3rd of next month
  Late payment fee: ₹50 + 1.5% monthly interest on outstanding amount
- Cash on Delivery: Available for orders up to ₹50,000, ₹50 COD fee, not available for jewelry/gold coins
- Gift Cards & SuperCoins: Redeemable on all products, non-refundable, valid for 1 year from purchase
- Wallet: Flipkart wallet (powered by PhonePe), max balance ₹10,000 for non-KYC users

## Flipkart Plus & SuperCoins
- 1 SuperCoin = ₹1 value, earned on every purchase (2 coins per ₹100 spent)
- Plus membership: Free for users with 200+ SuperCoins in last 12 months
- Benefits: Free/faster delivery, early access to sales, priority customer support (dedicated phone line)
- SuperCoin Store: Redeem for flight bookings, hotel stays, OTT subscriptions (Hotstar, Sony LIV), coupons

## Escalation Protocol
- Level 1 (Chat/Phone Agent): Resolves within 15 minutes — refund up to ₹3,000, reship orders
- Level 2 (Senior Agent): Order-level authority — refund up to ₹10,000, exception approvals
- Level 3 (Supervisor): Complaint-level authority — refund up to ₹50,000, policy exceptions, seller penalties
- Consumer Forum: If unresolved within 30 days → direct to Flipkart's Consumer Grievance Officer
  Flipkart's registered office: Bangalore, Karnataka. E-commerce consumer protection per Consumer Protection Act 2019

## Big Billion Days (BBD) Special Policies
- Extended return: 15 days for electronics (from 7/10), 45 days for fashion (from 30)
- Price protection: If price drops within 7 days of BBD purchase, credit the difference as SuperCoins
- Pre-order guarantee: Full refund if delivery delayed beyond promised date + ₹200 inconvenience credit
- Bank offers: Additional 10% instant discount with partner bank cards (max discount ₹2,000)
- Exchange bonus: Extra ₹3,000-₹5,000 on old smartphone exchange during BBD (over standard exchange value)
"""

token_count = count_tokens(ECOMMERCE_CATALOG)
print(f"📏 E-Commerce Catalog: {token_count} tokens")
print(f"   Gemini minimum: 1,024 → {'✅ EXCEEDS' if token_count >= 1024 else '❌ TOO SHORT'}")
print(f"   OpenAI minimum: 1,024 → {'✅ EXCEEDS' if token_count >= 1024 else '❌ TOO SHORT'}")

📏 E-Commerce Catalog: 1311 tokens
   Gemini minimum: 1,024 → ✅ EXCEEDS
   OpenAI minimum: 1,024 → ✅ EXCEEDS


## 5. OpenAI Automatic Prompt Caching — Banking Compliance Chatbot

**Business Scenario:** HDFC Bank's compliance department deploys an AI chatbot that helps officers check BSA/AML reporting requirements. Every request includes the full regulatory manual as a system prompt. OpenAI automatically caches this static prefix.

**How it works:**
- Caching activates automatically for prompts ≥ **1,024 tokens** — no code changes needed
- No write premium — first request costs the same as normal
- Cached tokens get **75% discount** on `gpt-4.1-mini` ($0.10/M vs $0.40/M)
- Check `usage.prompt_tokens_details.cached_tokens` in the response
- Cache lives **5-10 minutes** of inactivity (up to 1 hour max)

> 💡 We add a **5-second delay** between requests to let the cache propagate on OpenAI's servers.

In [7]:
# Different compliance queries — the long system prompt stays the same
compliance_queries = [
    "A customer deposited ₹9,50,000 in cash on Monday and ₹9,80,000 on Tuesday at two different HDFC branches in Mumbai. The customer's KYC shows ₹4 lakh annual income from a small retail shop. Is this structuring? What should we do?",
    "We received a ₹25,00,000 wire transfer from a company registered in Dubai to a newly opened current account in Gurugram branch. The account was opened 15 days ago. The beneficiary has not visited the branch. What is the risk level?",
    "A PEP (state minister's spouse) wants to open an NRI account and deposit ₹2 crore. They are unwilling to provide source of funds documentation. The relationship manager is pushing for quick account opening. Advise on CDD requirements.",
]

results = []
for i, query in enumerate(compliance_queries):
    response, elapsed = timed_call(
        openai_client.chat.completions.create,
        model="gpt-4.1-mini",
        temperature=0.2,     # Low temp for factual compliance responses
        messages=[
            {"role": "system", "content": BANKING_COMPLIANCE_SYSTEM},
            {"role": "user", "content": query}
        ]
    )
    stats = openai_cache_stats(response)
    results.append({"query_num": i+1, "elapsed": elapsed, **stats})
    display_result(f"OpenAI Request #{i+1}", stats, elapsed,
                   response.choices[0].message.content)

    # Allow cache to propagate before next request
    if i < len(compliance_queries) - 1:
        print("\n⏳ Waiting 5 seconds for cache propagation...")
        time.sleep(5)

# Summary table
print("\n📊 OpenAI Automatic Caching Summary")
print(f"{'Request':<10} {'Prompt Tkns':<14} {'Cached Tkns':<14} {'Cache Hit %':<14} {'Latency':<10}")
print("-" * 62)
for r in results:
    print(f"#{r['query_num']:<9} {r['prompt_tokens']:<14} {r['cached_tokens']:<14} {r['cache_hit_pct']:<14} {r['elapsed']:.2f}s")
print()
print("💡 Request #1 = cache MISS (cold start). Requests #2-3 should show cached_tokens > 0.")
print("   If cache_hit_pct is still 0%, try re-running this cell — caches may need longer to propagate.")


  OpenAI Request #1
  prompt_tokens                 : 1354
  completion_tokens             : 297
  cached_tokens                 : 0
  cache_hit_pct                 : 0.0
  latency_seconds               : 20.13s
  response_preview              : - **Risk Assessment**: HIGH  
- **Reporting Required**: Yes, Suspicious Transaction Report (STR)  
-...

⏳ Waiting 5 seconds for cache propagation...

  OpenAI Request #2
  prompt_tokens                 : 1349
  completion_tokens             : 296
  cached_tokens                 : 0
  cache_hit_pct                 : 0.0
  latency_seconds               : 8.20s
  response_preview              : - **Risk Assessment**: HIGH  
- **Reporting Required**: Yes, Suspicious Transaction Report (STR)  
-...

⏳ Waiting 5 seconds for cache propagation...

  OpenAI Request #3
  prompt_tokens                 : 1346
  completion_tokens             : 301
  cached_tokens                 : 1024
  cache_hit_pct                 : 76.1
  latency_seconds              

### 5.1 Multi-Turn Conversation Caching

**Business Scenario:** A compliance officer investigates a suspicious case across multiple conversation turns. As the conversation grows, the **entire prior message history** becomes the cached prefix. Each new turn only processes the latest user message from scratch — everything before it rides the cache.

This is where caching really shines: by Turn 3, the prompt easily exceeds 1,024 tokens with the system prompt + prior messages.

In [8]:
# Build a multi-turn conversation incrementally
conversation = [
    {"role": "system", "content": BANKING_COMPLIANCE_SYSTEM},
    {"role": "user", "content": "A customer account shows 5 cash deposits of ₹9,00,000 each over 3 consecutive days, all at different HDFC branches in Mumbai. The customer's profile shows a small grocery shop with ₹6 lakh annual turnover. Perform a risk assessment."},
]

# Turn 1 — cache miss, builds the initial cache
resp1, t1 = timed_call(openai_client.chat.completions.create,
                        model="gpt-4.1-mini", temperature=0.2, messages=conversation)
stats1 = openai_cache_stats(resp1)
assistant_reply_1 = resp1.choices[0].message.content

# Build Turn 2 — append assistant reply + new user question
conversation.append({"role": "assistant", "content": assistant_reply_1})
conversation.append({"role": "user", "content": "Good analysis. Now should we file the STR immediately or wait for the investigation to complete? Also check if this customer has any PEP connections or links to high-risk geographies."})
print("⏳ Waiting 5 seconds for cache propagation...")
time.sleep(5)

# Turn 2 — the growing conversation prefix should now be cached
resp2, t2 = timed_call(openai_client.chat.completions.create,
                        model="gpt-4.1-mini", temperature=0.2, messages=conversation)
stats2 = openai_cache_stats(resp2)
assistant_reply_2 = resp2.choices[0].message.content

# Build Turn 3
conversation.append({"role": "assistant", "content": assistant_reply_2})
conversation.append({"role": "user", "content": "Based on our analysis, draft a concise STR narrative covering all the red flags identified. Include the structuring pattern, income mismatch, and branch-hopping behavior."})
print("⏳ Waiting 5 seconds for cache propagation...")
time.sleep(5)

# Turn 3 — even more of the conversation is cached
resp3, t3 = timed_call(openai_client.chat.completions.create,
                        model="gpt-4.1-mini", temperature=0.2, messages=conversation)
stats3 = openai_cache_stats(resp3)

# Summary
print("\n📊 Multi-Turn Caching Progression")
print(f"{'Turn':<8} {'Prompt Tkns':<14} {'Cached Tkns':<14} {'Cache Hit %':<14} {'Latency':<10}")
print("-" * 60)
for i, (s, t) in enumerate([(stats1, t1), (stats2, t2), (stats3, t3)], 1):
    print(f"Turn {i:<5} {s['prompt_tokens']:<14} {s['cached_tokens']:<14} {s['cache_hit_pct']:<14} {t:.2f}s")
print()
print("💡 Watch cached_tokens grow with each turn. By Turn 2-3, most of the prompt is cache-read.")
print("   The cache grows as the conversation grows — only the latest user message is uncached.")

⏳ Waiting 5 seconds for cache propagation...
⏳ Waiting 5 seconds for cache propagation...

📊 Multi-Turn Caching Progression
Turn     Prompt Tkns    Cached Tkns    Cache Hit %    Latency   
------------------------------------------------------------
Turn 1     1348           1024           76.0           4.17s
Turn 2     1674           1536           91.8           3.94s
Turn 3     2028           1920           94.7           2.74s

💡 Watch cached_tokens grow with each turn. By Turn 2-3, most of the prompt is cache-read.
   The cache grows as the conversation grows — only the latest user message is uncached.


## 6. Anthropic Prompt Caching — Insurance Claims Processing

**Business Scenario:** Star Health Insurance deploys an AI claims adjudicator with a comprehensive reference document covering policy terms, clinical guidelines, and fraud detection rules. Multiple claims are evaluated against this same cached context.

**Anthropic Caching Modes:**
1. **Automatic caching** — Add `cache_control={"type": "ephemeral"}` at the **top level** of `messages.create()`. The system auto-places the breakpoint on the last cacheable block.
2. **Explicit breakpoints** — Place `cache_control` on individual content blocks for granular control (up to 4 breakpoints per request).

**Critical:** `claude-haiku-4-5` requires a minimum of **4,096 tokens** to activate caching. Our document is ~5,500 tokens — well above the threshold.

**Pricing:** Cache writes = 1.25× base input. Cache reads = **0.1×** base input (90% discount!)

In [9]:
# Verify our document exceeds the 4,096-token minimum for Haiku 4.5
doc_tokens = count_tokens(INSURANCE_HEALTHCARE_SYSTEM)
print(f"📏 Insurance+Healthcare document: {doc_tokens} tokens")
print(f"   Haiku 4.5 minimum: 4,096 → {'✅ EXCEEDS' if doc_tokens >= 4096 else '❌ TOO SHORT — increase document size!'}")

# Three different insurance claims evaluated against the same cached guidelines
claims_queries = [
    "Claim #SH-2024-78432: Mr. Rajesh Kumar, Gold Plan (active 3 years), admitted to Fortis Hospital Gurugram for laparoscopic cholecystectomy. Total bill: ₹3,45,000. Pre-auth obtained. 3-day stay, single AC room. Evaluate this claim.",
    "Claim #SH-2024-78501: Mrs. Priya Sharma, Silver Plan (active 8 months), emergency admission to Max Hospital Delhi for acute appendicitis. Total bill: ₹2,80,000. No pre-auth (emergency). 2-day stay, twin-sharing room. She was prescribed Tramadol and Sertraline together. Evaluate.",
    "Claim #SH-2024-78555: Mr. Amit Verma, Bronze Plan, admitted for bilateral knee replacement. Policy inception: 45 days ago. Total bill: ₹8,50,000. Pre-auth was denied but patient proceeded anyway. Evaluate and flag any issues.",
]

anthropic_results = []
for i, query in enumerate(claims_queries):
    response, elapsed = timed_call(
        anthropic_client.messages.create,
        model="claude-haiku-4-5",
        max_tokens=1024,
        temperature=0.2,
        cache_control={"type": "ephemeral"},   # ← Automatic caching
        system=INSURANCE_HEALTHCARE_SYSTEM,
        messages=[{"role": "user", "content": query}]
    )
    stats = anthropic_cache_stats(response)
    anthropic_results.append({"query_num": i+1, "elapsed": elapsed, **stats})
    display_result(f"Anthropic Claim #{i+1}", stats, elapsed,
                   response.content[0].text)

    if i < len(claims_queries) - 1:
        print("\n⏳ Waiting 3 seconds for cache propagation...")
        time.sleep(3)

# Summary
print("\n📊 Anthropic Automatic Caching Summary")
print(f"{'Request':<10} {'Input Tkns':<13} {'Cache Write':<14} {'Cache Read':<14} {'Latency':<10}")
print("-" * 61)
for r in anthropic_results:
    print(f"#{r['query_num']:<9} {r['input_tokens']:<13} {r['cache_creation_tokens']:<14} {r['cache_read_tokens']:<14} {r['elapsed']:.2f}s")
print()
print("💡 Claim #1: cache_creation_tokens > 0 (cache write — costs 1.25× base input)")
print("   Claims #2-3: cache_read_tokens > 0 (cache hit — costs only 0.1× base input = 90% savings!)")

📏 Insurance+Healthcare document: 5606 tokens
   Haiku 4.5 minimum: 4,096 → ✅ EXCEEDS

  Anthropic Claim #1
  input_tokens                  : 3
  output_tokens                 : 1024
  cache_creation_tokens         : 6863
  cache_read_tokens             : 0
  latency_seconds               : 10.79s
  response_preview              : # CLAIM EVALUATION REPORT
**Claim #SH-2024-78432**

---

## CLAIMANT DETAILS
| Field | Details |
|--...

⏳ Waiting 3 seconds for cache propagation...

  Anthropic Claim #2
  input_tokens                  : 3
  output_tokens                 : 1024
  cache_creation_tokens         : 6870
  cache_read_tokens             : 0
  latency_seconds               : 10.99s
  response_preview              : # CLAIM EVALUATION REPORT
**Claim #SH-2024-78501 | Mrs. Priya Sharma | Silver Plan**

---

## EXECUT...

⏳ Waiting 3 seconds for cache propagation...

  Anthropic Claim #3
  input_tokens                  : 3
  output_tokens                 : 1024
  cache_creation_tokens 

### 6.1 Anthropic Explicit Cache Breakpoints — Multi-Layer Caching

**Business Scenario:** Instead of caching the entire document as one block, we use **explicit breakpoints** to cache the system instruction and the reference document as separate layers. This gives us fine-grained control — changing the user query doesn't invalidate the cached document.

Explicit breakpoints are placed via `cache_control` on individual content blocks within the `system` parameter.

In [10]:
# Explicit breakpoints: cache the reference document block separately
# The system parameter is a LIST of content blocks — we mark the large document block for caching

explicit_results = []
physician_queries = [
    "A 58-year-old male Gold Plan member is admitted for NSTEMI. GRACE score is 155. His bill shows PCI with two stents at ₹5,20,000. He's also been prescribed Clopidogrel and Omeprazole together. Evaluate the claim and flag any issues.",
    "A 12-year-old child on Family Floater Platinum is admitted for febrile seizure. Weight: 35kg. Prescribed Paracetamol 500mg every 4 hours. 2-day stay with CT scan and EEG. Total bill: ₹1,85,000. Review clinical appropriateness.",
    "A 45-year-old female Silver Plan member claims ₹3,50,000 for laparoscopic cholecystectomy at a non-network hospital. She had the same procedure billed 5 months ago at a different hospital. Policy is 10 months old. Red flags?",
]

for i, query in enumerate(physician_queries):
    response, elapsed = timed_call(
        anthropic_client.messages.create,
        model="claude-haiku-4-5",
        max_tokens=1024,
        temperature=0.2,
        system=[
            {
                "type": "text",
                "text": "You are a senior claims adjudicator and clinical reviewer for Star Health Insurance."
            },
            {
                "type": "text",
                "text": INSURANCE_HEALTHCARE_SYSTEM,
                "cache_control": {"type": "ephemeral"}   # ← Explicit breakpoint on this block
            }
        ],
        messages=[{"role": "user", "content": query}]
    )
    stats = anthropic_cache_stats(response)
    explicit_results.append({"query_num": i+1, "elapsed": elapsed, **stats})
    display_result(f"Anthropic Explicit #{i+1}", stats, elapsed,
                   response.content[0].text)
    if i < len(physician_queries) - 1:
        print("\n⏳ Waiting 3 seconds...")
        time.sleep(3)

# Summary
print("\n📊 Anthropic Explicit Breakpoint Summary")
print(f"{'Request':<10} {'Input Tkns':<13} {'Cache Write':<14} {'Cache Read':<14} {'Latency':<10}")
print("-" * 61)
for r in explicit_results:
    print(f"#{r['query_num']:<9} {r['input_tokens']:<13} {r['cache_creation_tokens']:<14} {r['cache_read_tokens']:<14} {r['elapsed']:.2f}s")
print()
print("💡 Request #1 writes the document to cache. Requests #2-3 read from cache.")
print("   The explicit breakpoint only caches the large document block, not the short instruction prefix.")


  Anthropic Explicit #1
  input_tokens                  : 78
  output_tokens                 : 1024
  cache_creation_tokens         : 6797
  cache_read_tokens             : 0
  latency_seconds               : 11.23s
  response_preview              : # CLAIM EVALUATION REPORT
**Claim ID**: [To be assigned] | **Policyholder Age**: 58 years | **Plan**...

⏳ Waiting 3 seconds...

  Anthropic Explicit #2
  input_tokens                  : 83
  output_tokens                 : 1024
  cache_creation_tokens         : 0
  cache_read_tokens             : 6797
  latency_seconds               : 9.99s
  response_preview              : # CLAIMS REVIEW: FEBRILE SEIZURE — PEDIATRIC CASE

**Claim Reference Details:**
- **Policyholder**: ...

⏳ Waiting 3 seconds...

  Anthropic Explicit #3
  input_tokens                  : 71
  output_tokens                 : 1024
  cache_creation_tokens         : 0
  cache_read_tokens             : 6797
  latency_seconds               : 11.56s
  response_preview        

## 7. Google Gemini Explicit Caching — E-Commerce Catalog Q&A

**Business Scenario:** Flipkart caches their product catalog and policies as an explicit cache resource. Customer service queries reference this cached context, avoiding re-processing the full catalog with each question.

**Gemini Explicit Caching** is unique — it treats the cache as a **first-class API resource** with full CRUD operations:
- `client.caches.create()` — Create a named cache with TTL
- `client.caches.list()` — List all active caches
- `client.caches.update()` — Extend TTL
- `client.caches.delete()` — Delete to avoid storage costs

> ⚠️ Minimum 1,024 tokens for `gemini-2.5-flash`. Storage costs apply while the cache is alive.

In [ ]:
# Verify token count
catalog_tokens = count_tokens(ECOMMERCE_CATALOG)
print(f"📏 E-Commerce Catalog: {catalog_tokens} tokens")
print(f"   Gemini minimum: 1,024 → {'✅ EXCEEDS' if catalog_tokens >= 1024 else '❌ TOO SHORT'}")

# Create an explicit cache resource
cache = gemini_client.caches.create(
    model="gemini-2.5-flash",
    config=types.CreateCachedContentConfig(
        system_instruction="You are Flipkart's AI customer support assistant. Be helpful, concise, cite specific policies.",
        contents=[ECOMMERCE_CATALOG],
        display_name="flipkart-catalog-cache",
        ttl="600s",   # 10-minute TTL
    )
)

print(f"\n✅ Cache created!")
print(f"   Cache name : {cache.name}")
print(f"   Model      : {cache.model}")
print(f"   Token count: {cache.usage_metadata.total_token_count if cache.usage_metadata else 'N/A'}")
print(f"   Expires    : {cache.expire_time}")

### 7.1 Query the Cached Content

Each customer query references the cached catalog via `cached_content=cache.name`. The large catalog is read from cache — only the short customer question is processed fresh.

In [ ]:
customer_queries = [
    "I bought a Samsung Galaxy S24 during Big Billion Days and now the price dropped by ₹5,000 three days later. Can I get the difference back? What are my options?",
    "I ordered a Wakefit mattress and it arrived compressed. It's been 2 days and it still looks flat. Also my Flipkart Pay Later bill is overdue by 5 days — what are the charges?",
    "I received a damaged Nike shoe. It's been 25 days since delivery. Can I still return it under normal policy, or is there any BBD extended return window?",
]

gemini_results = []
for i, query in enumerate(customer_queries):
    response, elapsed = timed_call(
        gemini_client.models.generate_content,
        model="gemini-2.5-flash",
        contents=query,
        config=types.GenerateContentConfig(
            cached_content=cache.name,   # ← Reference the cache resource
        )
    )
    meta = response.usage_metadata
    stats = {
        "prompt_tokens": meta.prompt_token_count if meta else 0,
        "cached_tokens": meta.cached_content_token_count if meta and hasattr(meta, 'cached_content_token_count') else 0,
        "output_tokens": meta.candidates_token_count if meta else 0,
    }
    gemini_results.append({"query_num": i+1, "elapsed": elapsed, **stats})
    display_result(f"Gemini Cached Query #{i+1}", stats, elapsed,
                   response.text[:100] if response.text else "No text")

# Summary
print("\n📊 Gemini Explicit Caching Summary")
print(f"{'Query':<10} {'Prompt Tkns':<14} {'Cached Tkns':<14} {'Output Tkns':<14} {'Latency':<10}")
print("-" * 62)
for r in gemini_results:
    print(f"#{r['query_num']:<9} {r['prompt_tokens']:<14} {r['cached_tokens']:<14} {r['output_tokens']:<14} {r['elapsed']:.2f}s")
print()
print("💡 All queries should show cached_tokens ≈ catalog size. Each query only pays for the new question.")

### 7.2 Gemini Cache Lifecycle Management (CRUD Operations)

**Unique to Gemini:** Full developer control over cache resources. This is essential for cost management — active caches incur storage charges.

In [ ]:
# --- LIST all active caches ---
print("📋 Listing all active caches:")
for c in gemini_client.caches.list():
    token_count_val = c.usage_metadata.total_token_count if c.usage_metadata else "N/A"
    print(f"  • {c.name} | Tokens: {token_count_val} | Display: {c.display_name} | Expires: {c.expire_time}")

# --- UPDATE cache TTL (extend to 30 minutes) ---
updated_cache = gemini_client.caches.update(
    name=cache.name,
    config=types.UpdateCachedContentConfig(ttl="1800s")   # 30 min
)
print(f"\n🔄 Cache TTL extended → new expiry: {updated_cache.expire_time}")

# --- DELETE cache (cleanup to avoid ongoing storage costs) ---
gemini_client.caches.delete(name=cache.name)
print(f"\n🗑️ Cache deleted: {cache.name}")
print("   (In production, always delete caches when no longer needed to control storage costs)")

## 8. Cross-Provider Comparison — Same Document, Three Providers

**Business Scenario:** An enterprise team evaluates caching economics across providers using the same large reference document. We use the Insurance+Healthcare document (5,500+ tokens) which exceeds **all three** providers' minimums.

We send **two identical requests** per provider: first = cache miss, second = cache hit.

In [ ]:
# Shared query for all providers
COMPARISON_TASK = "A 62-year-old Gold Plan member is admitted to Apollo Hospital Chennai for emergency CABG surgery. Total bill is ₹7,20,000 with 12-day stay including 4 days in ICU. The patient is also on Warfarin and was given IV Heparin during admission. Evaluate the claim: clinical appropriateness, cost benchmarking, drug interaction review, and settlement recommendation."

comparison_results = []

# === OpenAI (2 requests) ===
print("🔵 OpenAI: Sending 2 requests with 5s gap...")
for attempt in range(2):
    resp, elapsed = timed_call(
        openai_client.chat.completions.create,
        model="gpt-4.1-mini", temperature=0.2,
        messages=[{"role": "system", "content": INSURANCE_HEALTHCARE_SYSTEM},
                  {"role": "user", "content": COMPARISON_TASK}]
    )
    stats = openai_cache_stats(resp)
    comparison_results.append({"provider": "OpenAI", "attempt": attempt+1, "elapsed": elapsed, **stats})
    print(f"  Request #{attempt+1}: cached_tokens={stats['cached_tokens']}, latency={elapsed:.2f}s")
    if attempt == 0:
        time.sleep(5)

# === Anthropic (2 requests) ===
print("\n🟠 Anthropic: Sending 2 requests with 3s gap...")
for attempt in range(2):
    resp, elapsed = timed_call(
        anthropic_client.messages.create,
        model="claude-haiku-4-5", max_tokens=1024, temperature=0.2,
        cache_control={"type": "ephemeral"},
        system=INSURANCE_HEALTHCARE_SYSTEM,
        messages=[{"role": "user", "content": COMPARISON_TASK}]
    )
    stats = anthropic_cache_stats(resp)
    comparison_results.append({"provider": "Anthropic", "attempt": attempt+1, "elapsed": elapsed, **stats})
    print(f"  Request #{attempt+1}: cache_write={stats['cache_creation_tokens']}, cache_read={stats['cache_read_tokens']}, latency={elapsed:.2f}s")
    if attempt == 0:
        time.sleep(3)

# === Gemini (2 requests — create a fresh cache for comparison) ===
print("\n🟢 Gemini: Creating cache + sending 2 queries...")
gemini_compare_cache = gemini_client.caches.create(
    model="gemini-2.5-flash",
    config=types.CreateCachedContentConfig(
        system_instruction="You are a senior claims adjudicator for Star Health Insurance.",
        contents=[INSURANCE_HEALTHCARE_SYSTEM],
        display_name="comparison-cache",
        ttl="300s",
    )
)
for attempt in range(2):
    resp, elapsed = timed_call(
        gemini_client.models.generate_content,
        model="gemini-2.5-flash",
        contents=COMPARISON_TASK,
        config=types.GenerateContentConfig(cached_content=gemini_compare_cache.name)
    )
    meta = resp.usage_metadata
    cached_val = meta.cached_content_token_count if meta and hasattr(meta, 'cached_content_token_count') else 0
    stats = {"prompt_tokens": meta.prompt_token_count if meta else 0, "cached_tokens": cached_val}
    comparison_results.append({"provider": "Gemini", "attempt": attempt+1, "elapsed": elapsed, **stats})
    print(f"  Request #{attempt+1}: cached_tokens={cached_val}, latency={elapsed:.2f}s")

# Cleanup Gemini cache
gemini_client.caches.delete(name=gemini_compare_cache.name)
print("  (cache cleaned up)")

# Display comparison table
print("\n📊 Cross-Provider Caching Comparison")
print(f"{'Provider':<12} {'Attempt':<10} {'Latency':<10} {'Cache Metrics'}")
print("-" * 70)
for r in comparison_results:
    if r["provider"] == "OpenAI":
        cache_info = f"cached_tokens={r.get('cached_tokens', 0)}, hit={r.get('cache_hit_pct', 0)}%"
    elif r["provider"] == "Anthropic":
        cache_info = f"write={r.get('cache_creation_tokens', 0)}, read={r.get('cache_read_tokens', 0)}"
    else:
        cache_info = f"cached={r.get('cached_tokens', 0)}"
    print(f"{r['provider']:<12} #{r['attempt']:<9} {r['elapsed']:.2f}s     {cache_info}")

## 9. Cache-Aware Prompt Design — Good vs Bad Patterns

The **golden rule** of prompt caching: *static content first, variable content last*.

We demonstrate this with OpenAI: same system prompt, but one version embeds a changing timestamp in the prefix (breaking the cache), while the other keeps the prefix stable.

In [ ]:
import datetime

# === PATTERN 1: CACHE-FRIENDLY (static prefix) ===
print("✅ PATTERN 1: Cache-Friendly — Static system prompt, dynamic user query only")
print("-" * 70)

friendly_results = []
for query in [
    "What is the penalty for non-compliance with PMLA Section 12? How much can RBI fine a bank for KYC violations?",
    "Explain the difference between a CTR and an STR. When do we file both for the same transaction?",
]:
    resp, elapsed = timed_call(
        openai_client.chat.completions.create,
        model="gpt-4.1-mini", temperature=0.3,
        messages=[
            {"role": "system", "content": BANKING_COMPLIANCE_SYSTEM},  # ← Same prefix every time
            {"role": "user", "content": query}
        ]
    )
    stats = openai_cache_stats(resp)
    friendly_results.append(stats)
    print(f"  Query: {query[:65]}...")
    print(f"  Cached: {stats['cached_tokens']} tokens ({stats['cache_hit_pct']}%)  Latency: {elapsed:.2f}s")
    time.sleep(5)

print()

# === PATTERN 2: CACHE-BREAKING (dynamic timestamp in prefix) ===
print("❌ PATTERN 2: Cache-Breaking — Dynamic timestamp injected in system prompt")
print("-" * 70)

broken_results = []
for query in [
    "What is the penalty for non-compliance with PMLA Section 12? How much can RBI fine a bank for KYC violations?",
    "Explain the difference between a CTR and an STR. When do we file both for the same transaction?",
]:
    # BAD PRACTICE: Timestamp changes the prefix on every request, busting the cache
    dynamic_system = f"[Request timestamp: {datetime.datetime.now().isoformat()}]\n\n{BANKING_COMPLIANCE_SYSTEM}"
    resp, elapsed = timed_call(
        openai_client.chat.completions.create,
        model="gpt-4.1-mini", temperature=0.3,
        messages=[
            {"role": "system", "content": dynamic_system},   # ← Different prefix every time!
            {"role": "user", "content": query}
        ]
    )
    stats = openai_cache_stats(resp)
    broken_results.append(stats)
    print(f"  Query: {query[:65]}...")
    print(f"  Cached: {stats['cached_tokens']} tokens ({stats['cache_hit_pct']}%)  Latency: {elapsed:.2f}s")
    time.sleep(5)

print()
print("💡 KEY TAKEAWAYS:")
print("   1. Never embed timestamps, session IDs, UUIDs, or counters in system prompts")
print("   2. Move all dynamic content (user context, session info) to the USER message")
print("   3. Keep tool definitions and their ORDER identical across requests")
print("   4. If you must include dynamic metadata, place it AFTER all static content")

## 10. Cost & Timing Analysis

In [ ]:
# Pricing per million tokens (Feb 2026)
PRICING = {
    "gpt-4.1-mini":     {"input": 0.40, "cached_input": 0.10, "output": 1.60, "cache_write": 0.40},
    "claude-haiku-4-5": {"input": 1.00, "cached_input": 0.10, "output": 5.00, "cache_write": 1.25},
    "gemini-2.5-flash": {"input": 0.15, "cached_input": 0.0375, "output": 0.60, "cache_write": 0.15},
}

print("📊 Provider Pricing Quick Reference (per million tokens)")
print(f"{'Model':<25} {'Input':<10} {'Cache Write':<14} {'Cache Read':<14} {'Output':<10}")
print("-" * 73)
for model, p in PRICING.items():
    print(f"{model:<25} ${p['input']:<9} ${p['cache_write']:<13} ${p['cached_input']:<13} ${p['output']}")

print()
print("💰 ESTIMATED COST FOR THIS LAB RUN")
print("=" * 55)
cost_items = {
    "OpenAI (gpt-4.1-mini)":     {"est": 0.015, "note": "~12 calls × ~1,800 prompt tokens"},
    "Anthropic (haiku-4-5)":     {"est": 0.035, "note": "~6 calls × ~6,000 prompt tokens"},
    "Gemini (gemini-2.5-flash)": {"est": 0.008, "note": "~7 calls × ~1,800+ tokens + cache storage"},
}
total = 0
for comp, info in cost_items.items():
    print(f"  {comp:<35s} ${info['est']:.3f}  ({info['note']})")
    total += info["est"]
print("-" * 55)
print(f"  {'TOTAL':<35s} ${total:.3f}")

print()
print("📈 PRODUCTION SAVINGS EXAMPLE: Banking Compliance Chatbot")
print("   Document size: ~1,800 tokens system prompt")
print("   Queries per day: 500")
print("   Without caching: 500 × 1,800 tokens × $0.40/M = $0.36/day")
print("   With caching:    500 × 1,800 tokens × $0.10/M = $0.09/day (75% savings)")
print("   Annual savings: ~$98.55 on input tokens alone for a single chatbot")
print("   At enterprise scale (50 chatbots): ~$4,927/year")

---

## 11. Conclusion & Next Steps

### What We Covered

| Task | Key Takeaway |
|------|-------------|
| **Token Verification** | Always count tokens BEFORE attempting caching — each provider has different minimums (1,024 for OpenAI/Gemini, 4,096 for Haiku 4.5) |
| **OpenAI Automatic** | Zero-config for ≥ 1,024 tokens; no write premium; monitor via `cached_tokens`; 5-10 min TTL |
| **OpenAI Multi-Turn** | Conversation history becomes the cached prefix — cache grows with each turn |
| **Anthropic Automatic** | Single `cache_control={"type": "ephemeral"}` enables auto-caching; 90% discount on reads; 25% premium on writes |
| **Anthropic Explicit** | Place `cache_control` on individual content blocks for multi-layer caching control (up to 4 breakpoints) |
| **Gemini Explicit** | Resource-based CRUD API — create, list, update TTL, delete; configurable TTL; storage costs while alive |
| **Cross-Provider** | Same document cached across all providers — each has different economics and control models |
| **Design Patterns** | Golden rule: static first, dynamic last; never embed timestamps/IDs in system prompts |

### Try on Your Own

1. **OpenAI Extended Retention:** Add `prompt_cache_retention="24h"` and `prompt_cache_key="compliance-bot"` to your requests — measure improvement in hit rates across longer intervals
2. **Anthropic 1-Hour TTL:** Use `cache_control={"type": "ephemeral", "ttl": "3600"}` and compare write cost (2× base) vs the 5-min default (1.25× base) — when does the higher write cost pay off?
3. **Gemini Multimodal Caching:** Upload a PDF via `gemini_client.files.upload()`, create an explicit cache with the file content, and ask multiple questions about it
4. **Cache Warming:** Implement the Thomson Reuters pattern: send a minimal "warm-up" request to populate the cache before firing parallel queries (avoids the race condition where parallel requests each create their own cache)
5. **RAG + Caching Hybrid:** Place a stable system prompt and few-shot examples at the start of your messages, then append dynamic RAG-retrieved documents. The stable prefix caches even as the RAG content changes
6. **Break-Even Calculator:** Build a function `cache_roi(provider, prefix_tokens, queries_per_day, avg_gap_seconds)` that calculates whether caching is cost-effective given your traffic pattern

---

**Next:** Apply prompt caching patterns in your LangChain/LangGraph agent workflows for production cost optimization 🚀

---
## 11. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Automatic vs explicit caching** | OpenAI auto-caches stable prefixes ≥1024 tokens. Anthropic & Gemini require explicit breakpoints / cache objects. |
| **Cache breakpoints (Anthropic)** | Mark `cache_control` on system / user / tool blocks — up to 4 breakpoints; each becomes a cacheable layer. |
| **Gemini explicit cache** | Create a cache object once (`ttl=3600s`), reference by ID across many queries — best for >32K static reference. |
| **Token thresholds** | OpenAI: 1024 · Anthropic Sonnet: 1024, Haiku: 4096 · Gemini: 32768. Below threshold = no cache, full price. |
| **Cache-friendly design** | Stable prefix → dynamic suffix. Avoid timestamps, random IDs, or per-request-varying instructions before the breakpoint. |
| **Cost math** | Cached input typically billed at 10–25% of base rate. Long static system + many short user queries = the textbook win. |

**Next Lab:** Module 3 (Day 3) — LangChain Deep Dive · Chains, Modern Memory, LCEL Composition 🔗


## 12. Stretch Exercise (Optional)

1. Build a 1000-call benchmark: same system prompt, 1000 different user queries. Measure cache hit rate and total cost on each provider.
2. Refactor the *cache-hostile* prompt example so the dynamic content moves to the *end* of the system block. Verify the cache hit rate jumps.
3. On Anthropic, place 2 breakpoints (one for system, one for retrieved RAG context). Compare cost vs single-breakpoint.
4. On Gemini, write a Lambda that automatically refreshes a cache before TTL expiry — production lifecycle pattern.
5. Compute the breakeven: at how many queries does the cache-creation overhead pay off vs uncached?


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. How does Anthropic's caching differ from OpenAI's automatic caching?**

*Hint:* Explicit vs implicit.

*Answer sketch:* OpenAI **auto-caches** any prompt prefix ≥1024 tokens that's seen recently — no API change. Anthropic requires you to **explicitly mark** cache breakpoints with `cache_control: {type: 'ephemeral'}` on the message block — up to 4 breakpoints per request, each cacheable independently. Anthropic's pattern gives more control (multi-layer caching: tools + system + RAG context) but requires deliberate prompt design.

---

**Q2. What is a cache breakpoint, and where do you place them?**

*Hint:* Think layers of stable-vs-changing content.

*Answer sketch:* A cache breakpoint marks the *end* of a cacheable prefix. Place one after stable content that you want cached: end of `tools=[...]`, end of system prompt, end of retrieved RAG chunks. Everything *before* the breakpoint is cached; everything *after* is uncached and freshly evaluated. Use multiple breakpoints when different layers update at different rates (tools rarely change → cache long; RAG chunks per-conversation → cache short).

---

**Q3. Minimum token thresholds for caching across the 3 providers — why do they exist?**

*Hint:* Cache infrastructure has overhead.

*Answer sketch:* OpenAI: 1024. Anthropic Sonnet: 1024. Anthropic Haiku: 4096. Gemini explicit cache: 32768. Below threshold, the cache hash + lookup overhead exceeds the savings, so providers refuse to cache. Practical implication: short prompts (≤1K tokens) don't benefit from caching at all on any provider — caching is for long static system prompts and RAG contexts.

---

**Q4. What kinds of prompts are cache-friendly vs cache-hostile?**

*Hint:* Stability of the prefix.

*Answer sketch:* Cache-friendly: long stable system prompt + tools + few-shot examples → short user query. Cache-hostile: timestamp injected at the top, random shuffled few-shot examples, user-id concatenated into the system prompt, or any per-request randomness in the prefix. Refactor to push all dynamic content to the *end* of the prompt and the cache fires.

---

**Q5. Cache TTLs: when does a cache get evicted, and how do you design around it?**

*Hint:* Different providers, different rules.

*Answer sketch:* OpenAI: ~5–10 min idle TTL (auto). Anthropic ephemeral: 5 minutes default, 1-hour option. Gemini: caller-specified TTL via `ttl` parameter (default 1h, max 24h+). Design: send a 'keepalive' query every few minutes for chat, or for Gemini explicitly extend TTL via `update_cache(ttl=...)`. For RAG with rotating context, accept that cache misses happen at TTL expiry and budget for it.

---

**Q6. Cost math: prompt with 50K static system + 500 user — savings from caching?**

*Hint:* Compare cached input rate vs base rate.

*Answer sketch:* Take Anthropic Haiku at ~$0.80/M input · ~$0.20/M cached input. Uncached call: 50500 × $0.80/M ≈ $0.040. Cached call: 50000 × $0.20/M + 500 × $0.80/M ≈ $0.0104. Per-call savings ~75%. Over 1M calls/day = ~$30K/month difference. The breakeven against the one-time cache-creation cost is reached in under 5 calls.

---

**Q7. How does Gemini's explicit caching API differ from Anthropic's `cache_control`?**

*Hint:* Cache as a first-class object vs a per-request flag.

*Answer sketch:* Anthropic: in-line — pass `cache_control: {type:'ephemeral'}` on a message block, cache lives implicitly with that prompt. Gemini: out-of-line — call `caches.create(model, contents, ttl)` to get a cache ID, then reference it in subsequent `generate_content(cached_content=cache_id, contents=user_query)`. Gemini's pattern is better for many queries against the *same* large reference (32K+ tokens, expensive to recreate). Anthropic's is better for chat where the system prompt + tools are reused.

